# Step 5: the US overnight move

- **Feature:** the S&P 500's return in the last US session before each Asian day.
- **Two targets:**
  - **close-to-close:** not tradeable. The US move happens after Asia's previous close, so you'd have to buy before the news.
  - **open-to-close:** tradeable. See the US move, buy at the Asian open, sell at the close.
- **Models:** logistic regression and random forest, kept simple so the result is about the feature.

In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

from classifiers import CLASSIFIERS, fit_classifier, model_table, overview
from data import load_all, load_us
from evaluation import majority_direction, money_test
from features import make_dataset, make_features, make_us_features, up_close_to_close, up_open_to_close
from models import pick_by_log_loss, pooled, walk_forward

us = load_us()
markets = {market: df[:"2022"] for market, df in load_all().items()}  # the US file ends in 2022

def us_feature(df):
    return make_us_features(df, us)

## Data check: are the opens real?

% of days where the open is just a copy of yesterday's close. Those days are left out of the open-to-close target.

In [2]:
fake_open = {}
for market, df in markets.items():
    copy_of_close = df["Open"] == df["Close"].shift(1)
    fake_open[market] = 100 * copy_of_close.groupby(df.index.year).mean()
pd.DataFrame(fake_open).T.round(0)

Date,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022
Indonesia,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,45.0,0.0,26.0
Malaysia,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Philippines,6.0,6.0,5.0,6.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Singapore,89.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Thailand,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
India,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


How closely the US move lines up with each part of the Asian day (correlation, real opens only):

In [3]:
def correlations(df, skip_year=None):
    """Correlation of the US move with each part of the Asian day."""
    us_ret = us_feature(df)["us_ret"]
    gap = np.log(df["Open"] / df["Close"].shift(1))  # yesterday's close -> today's open
    day = np.log(df["Close"] / df["Open"])           # today's open -> close
    keep = (df["Open"] != df["Close"].shift(1)) & (df.index.year != skip_year)
    return {
        "close-to-close": us_ret[keep].corr(df["ret"][keep]),
        "overnight gap": us_ret[keep].corr(gap[keep]),
        "open-to-close": us_ret[keep].corr(day[keep]),
    }

pd.DataFrame({market: correlations(df) for market, df in markets.items()}).T.round(3)

,close-to-close,overnight gap,open-to-close
Indonesia,0.271,0.556,0.059
Malaysia,0.311,0.518,0.051
Philippines,0.315,0.473,0.117
Singapore,0.347,0.599,-0.034
Thailand,0.241,0.500,-0.088
India,0.272,0.494,-0.025


## Models

Six walk-forward folds, as before. About 3 minutes.

In [4]:
MODELS = {
    "Logistic regression": CLASSIFIERS["Logistic regression"],
    "Random forest": [RandomForestClassifier(n_estimators=300, min_samples_leaf=leaf, random_state=0, n_jobs=-1)
                      for leaf in [10, 50, 200]],
}
TARGETS = {"close-to-close": up_close_to_close, "open-to-close": up_open_to_close}

results = {}
for target, make_y in TARGETS.items():
    for name, models in MODELS.items():
        results[target, name] = {market: walk_forward(df, fit_classifier, models, make_x=us_feature, make_y=make_y)
                                 for market, df in markets.items()}

## The two targets side by side

All six test years pooled.

In [5]:
COLUMNS = ["accuracy %", "fair bar %", "p vs fair", "AUC", "MCC"]
for name in MODELS:
    print(name)
    display(pd.concat({target: model_table(results[target, name])[COLUMNS] for target in TARGETS}, axis=1).round(3))

Logistic regression


close-to-close                                    open-to-close  \
                accuracy % fair bar % p vs fair    AUC    MCC    accuracy %   
Indonesia           57.996     52.917       0.0  0.621  0.153        48.327   
Malaysia            59.945     49.761       0.0  0.648  0.206        49.795   
Philippines         60.465     51.778       0.0  0.644  0.210        55.921   
Singapore           61.421     50.930       0.0  0.657  0.228        49.369   
Thailand            57.790     52.437       0.0  0.619  0.158        54.770   
India               60.784     54.564       0.0  0.637  0.197        52.436   

                                                
            fair bar % p vs fair    AUC    MCC  
Indonesia       49.027     0.163  0.504 -0.047  
Malaysia        48.904     0.432  0.550  0.015  
Philippines     50.171     0.000  0.580  0.128  
Singapore       49.767     0.418  0.478 -0.026  
Thailand        54.358     0.745  0.528  0.050  
India           52.503     1.000  0.494 -0.006

Random forest


close-to-close                                    open-to-close  \
                accuracy % fair bar % p vs fair    AUC    MCC    accuracy %   
Indonesia           56.692     52.917     0.019  0.603  0.123        50.661   
Malaysia            61.039     49.761     0.000  0.637  0.222        54.521   
Philippines         60.670     51.778     0.000  0.637  0.211        55.989   
Singapore           61.421     50.930     0.000  0.653  0.229        49.169   
Thailand            57.310     52.437     0.001  0.609  0.138        52.642   
India               60.581     54.564     0.000  0.631  0.195        50.338   

                                                
            fair bar % p vs fair    AUC    MCC  
Indonesia       49.027     0.344  0.509  0.020  
Malaysia        48.904     0.002  0.546  0.093  
Philippines     50.171     0.001  0.568  0.121  
Singapore       49.767     0.749  0.488 -0.016  
Thailand        54.358     0.308  0.512  0.033  
India           52.503     0.075  0.485 -0.028

All 36 folds (6 markets x 6 test years). "Significantly" = McNemar p < 0.05 against the fair bar.

In [6]:
overview({f"{target} | {name}": preds for (target, name), preds in results.items()}).round(3)

,mean AUC,above fair bar,below fair bar,significantly above,significantly below,above stretch bar,one-answer folds
close-to-close | Logistic regression,0.639,35.0,0.0,15.0,0.0,35.0,0.0
close-to-close | Random forest,0.629,34.0,2.0,12.0,0.0,34.0,0.0
open-to-close | Logistic regression,0.528,10.0,8.0,3.0,0.0,7.0,17.0
open-to-close | Random forest,0.521,21.0,10.0,3.0,0.0,14.0,4.0


Test days per market. Open-to-close has fewer where some opens are fake:

In [7]:
pd.DataFrame({target: {market: len(p) for market, p in results[target, "Logistic regression"].items()}
              for target in TARGETS})

,close-to-close,open-to-close
Indonesia,1457,1285
Malaysia,1463,1460
Philippines,1462,1461
Singapore,1506,1505
Thailand,1457,1457
India,1479,1478


## Full tables (step 3 style)

**one-answer folds** = test years (out of 6) where the model gave the same answer every day (all up, or all down).

In [8]:
for (target, name), preds_by_market in results.items():
    print(f"{target} | {name}")
    display(model_table(preds_by_market).round(3))

close-to-close | Logistic regression


,accuracy %,MCC,AUC,fair bar %,p vs fair,same as yesterday %,p vs same as yesterday,called up %,one-answer folds
Indonesia,57.996,0.153,0.621,52.917,0.0,49.485,0.0,76.802,0.0
Malaysia,59.945,0.206,0.648,49.761,0.0,49.282,0.0,62.064,0.0
Philippines,60.465,0.210,0.644,51.778,0.0,48.427,0.0,66.826,0.0
Singapore,61.421,0.228,0.657,50.930,0.0,49.602,0.0,53.652,0.0
Thailand,57.790,0.158,0.619,52.437,0.0,50.995,0.0,79.959,0.0
India,60.784,0.197,0.637,54.564,0.0,54.767,0.0,74.037,0.0


close-to-close | Random forest


,accuracy %,MCC,AUC,fair bar %,p vs fair,same as yesterday %,p vs same as yesterday,called up %,one-answer folds
Indonesia,56.692,0.123,0.603,52.917,0.019,49.485,0.000,63.281,0.0
Malaysia,61.039,0.222,0.637,49.761,0.000,49.282,0.000,54.682,0.0
Philippines,60.670,0.211,0.637,51.778,0.000,48.427,0.000,59.508,0.0
Singapore,61.421,0.229,0.653,50.930,0.000,49.602,0.000,48.207,0.0
Thailand,57.310,0.138,0.609,52.437,0.001,50.995,0.000,68.771,0.0
India,60.581,0.195,0.631,54.564,0.000,54.767,0.001,63.151,0.0


open-to-close | Logistic regression


,accuracy %,MCC,AUC,fair bar %,p vs fair,same as yesterday %,p vs same as yesterday,called up %,one-answer folds
Indonesia,48.327,-0.047,0.504,49.027,0.163,48.171,0.969,97.432,5.0
Malaysia,49.795,0.015,0.550,48.904,0.432,50.000,0.941,84.041,2.0
Philippines,55.921,0.128,0.580,50.171,0.000,48.597,0.000,70.294,1.0
Singapore,49.369,-0.026,0.478,49.767,0.418,49.302,1.000,2.525,4.0
Thailand,54.770,0.050,0.528,54.358,0.745,47.975,0.000,16.198,0.0
India,52.436,-0.006,0.494,52.503,1.000,48.444,0.029,0.474,5.0


open-to-close | Random forest


,accuracy %,MCC,AUC,fair bar %,p vs fair,same as yesterday %,p vs same as yesterday,called up %,one-answer folds
Indonesia,50.661,0.020,0.509,49.027,0.344,48.171,0.213,65.214,0.0
Malaysia,54.521,0.093,0.546,48.904,0.002,50.000,0.015,54.521,0.0
Philippines,55.989,0.121,0.568,50.171,0.001,48.597,0.000,59.685,0.0
Singapore,49.169,-0.016,0.488,49.767,0.749,49.302,0.971,41.661,1.0
Thailand,52.642,0.033,0.512,54.358,0.308,47.975,0.017,38.092,0.0
India,50.338,-0.028,0.485,52.503,0.075,48.444,0.321,20.568,3.0


# Step 5b: checking the two exceptions

Open-to-close showed an edge in the Philippines (both models) and Malaysia (random forest only). Is it something you could trade, or a side effect of how the index's opening number is recorded?

## 1. Is it a few huge days? Remove 2020

Correlations without 2020:

In [9]:
pd.DataFrame({market: correlations(df, skip_year=2020) for market, df in markets.items()}).T.round(3)

,close-to-close,overnight gap,open-to-close
Indonesia,0.272,0.556,0.063
Malaysia,0.324,0.453,0.109
Philippines,0.338,0.462,0.187
Singapore,0.366,0.629,-0.040
Thailand,0.223,0.524,-0.081
India,0.259,0.516,-0.034


Open-to-close results, pooled over the five test years other than 2020:

In [10]:
EXCEPTIONS = ["Philippines", "Malaysia"]

no_2020 = {}
for market in EXCEPTIONS:
    for name in MODELS:
        p = results["open-to-close", name][market]
        no_2020[market, name] = pooled(p[p["year"] != 2020])
pd.DataFrame(no_2020).T[COLUMNS].round(3)

accuracy %  fair bar %  p vs fair    AUC  \
Philippines Logistic regression      56.475      50.082      0.000  0.590   
            Random forest            56.557      50.082      0.000  0.573   
Malaysia    Logistic regression      49.794      49.052      0.523  0.558   
            Random forest            54.988      49.052      0.003  0.550   

                                   MCC  
Philippines Logistic regression  0.142  
            Random forest        0.133  
Malaysia    Logistic regression  0.015  
            Random forest        0.102

The tables above only leave 2020 out of the scoring. Here 2020 is also removed from training and tuning. The 2021 fold tunes on 2019, because its usual tune year is gone.

In [11]:
FOLDS = {2017: 2016, 2018: 2017, 2019: 2018, 2021: 2019, 2022: 2021}  # test year: tune year

def walk_forward_without_2020(df, models):
    X, y = make_dataset(df, us_feature, up_open_to_close)
    X, y = X[X.index.year != 2020], y[y.index.year != 2020]
    up_yesterday = up_open_to_close(df).shift(1).eq(True)
    years = X.index.year
    folds = []
    for test_year, tune_year in FOLDS.items():
        train, tune, test = years < tune_year, years == tune_year, years == test_year
        best = pick_by_log_loss(fit_classifier, models, X[train], y[train], X[tune], y[tune])
        before_test = years < test_year
        predict = fit_classifier(X[before_test], y[before_test], best)
        folds.append(pd.DataFrame({
            "year": test_year,
            "p_up": predict(X[test]),
            "up": y[test],
            "fair_call": majority_direction(y[before_test]),
            "same_as_yesterday": up_yesterday[X.index[test]],
        }, index=X.index[test]))
    return pd.concat(folds)

fully_removed = {}
for market in EXCEPTIONS:
    for name, models in MODELS.items():
        fully_removed[market, name] = pooled(walk_forward_without_2020(markets[market], models))
pd.DataFrame(fully_removed).T[COLUMNS].round(3)

accuracy %  fair bar %  p vs fair    AUC  \
Philippines Logistic regression      56.475      50.082      0.000  0.590   
            Random forest            56.557      50.082      0.000  0.575   
Malaysia    Logistic regression      49.711      49.052      0.575  0.533   
            Random forest            54.988      49.052      0.003  0.547   

                                   MCC  
Philippines Logistic regression  0.135  
            Random forest        0.133  
Malaysia    Logistic regression  0.012  
            Random forest        0.102

## 2. Is it the US news arriving late? Remove the US feature

Same target and models, but with step 3's features (the market's own last 5 returns and yesterday's direction) instead of the US move.

In [12]:
no_us = {}
for market in EXCEPTIONS:
    for name, models in MODELS.items():
        p = walk_forward(markets[market], fit_classifier, models, make_x=make_features, make_y=up_open_to_close)
        no_us[market, name] = pooled(p)
pd.DataFrame(no_us).T[COLUMNS].round(3)

accuracy %  fair bar %  p vs fair    AUC  \
Philippines Logistic regression      49.487      50.171      0.426  0.488   
            Random forest            51.335      50.171      0.478  0.512   
Malaysia    Logistic regression      48.699      48.904      0.856  0.493   
            Random forest            49.384      48.904      0.668  0.514   

                                   MCC  
Philippines Logistic regression -0.023  
            Random forest        0.027  
Malaysia    Logistic regression -0.014  
            Random forest        0.005

## 3. A price you could actually trade

Checked on Yahoo for 2017-2022:
- **Philippines:** the FMETF fund isn't on Yahoo.
- **Malaysia:** the FTSE Bursa Malaysia KLCI ETF (`0820EA.KL`) is, but it had no trades on 41% of days and its open equals its close on 82%. Its opens can't show what a buyer at the open would pay.
- **KLCI futures:** none found. Yahoo's search returned nothing relevant, and `FKLI=F`, `FKLI.KL`, `KLCI=F` and `FBMKLCI=F` returned no data.

So the money test below uses the index itself, which no one can buy at its opening value.

## 4. Money test

Buy at the open and sell at the close on days the model says up; otherwise stay out. Compared with buying at the open every day. Total return over the six test years, at different round-trip trading costs.

In [13]:
COSTS = [0, 0.001, 0.0025, 0.005]  # round trip: 0%, 0.1%, 0.25%, 0.5%

rows = {}
for market in EXCEPTIONS:
    for name in MODELS:
        p = results["open-to-close", name][market]
        df = markets[market].loc[p.index]
        day_return = df["Close"] / df["Open"] - 1
        calls_up = p["p_up"] > 0.5
        row = {"days in": calls_up.sum(), "average day in %": 100 * day_return[calls_up].mean()}
        for cost in COSTS:
            model, every_day = money_test(day_return, calls_up, cost)
            row[f"model % (cost {cost:.2%})"] = 100 * (model - 1)
            row[f"every day % (cost {cost:.2%})"] = 100 * (every_day - 1)
        rows[market, name] = row
pd.DataFrame(rows).round(2)

Philippines                \
                         Logistic regression Random forest   
days in                              1027.00        872.00   
average day in %                        0.09          0.12   
model % (cost 0.00%)                  127.69        163.00   
every day % (cost 0.00%)                3.44          3.44   
model % (cost 0.10%)                  -18.45         10.02   
every day % (cost 0.10%)              -76.02        -76.02   
model % (cost 0.25%)                  -82.55        -70.28   
every day % (cost 0.25%)              -97.33        -97.33   
model % (cost 0.50%)                  -98.67        -96.66   
every day % (cost 0.50%)              -99.93        -99.93   

                                    Malaysia                
                         Logistic regression Random forest  
days in                              1227.00        796.00  
average day in %                        0.00          0.05  
model % (cost 0.00%)                   -2.37         43.27  
every day % (cost 0.00%)              -10.99        -10.99  
model % (cost 0.10%)                  -71.40        -35.37  
every day % (cost 0.10%)              -79.35        -79.35  
model % (cost 0.25%)                  -95.47        -80.45  
every day % (cost 0.25%)              -97.70        -97.70  
model % (cost 0.50%)                  -99.79        -97.34  
every day % (cost 0.50%)              -99.94        -99.94

**average day in %** is also the break-even cost: a round-trip cost above it wipes out the model's edge.